# Intro 2：TabularX 多分类任务 —— Iris 数据集

本教程演示 TabularX 在多分类任务上的使用。Iris 数据集包含 3 个品种的鸢尾花，4 个特征。


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from sklearn.datasets import load_iris
import tabularx as tbx

print("TabularX 已就绪")


TabularX 已就绪


In [2]:
data = load_iris(as_frame=True)
X, y = data.data, data.target

print(f"特征维度: {X.shape}")
print(f"样本数: {len(X)}")
print(f"特征数: {len(X.columns)}")
print(f"目标分布:\n{y.value_counts()}")


特征维度: (150, 4)
样本数: 150
特征数: 4
目标分布:
target
0    50
1    50
2    50
Name: count, dtype: int64


## 创建 AutoEstimator

多分类任务传 `task="classification"`，框架自动识别为 multiclass。


In [3]:
ae = tbx.AutoEstimator(task="classification",
                     output_dir="./intro2-output",
                     pdf_report=True)

# 多分类搜索轮数少一些即可
ae.param.model_search.set_param("n_trials", 20)
ae.param.model_search.set_param("top_k_models", 2)
ae.param.shap.set_param("top_k_features", 4)
ae.param.split.set_param("test_size", 0.2)
ae.param.split.set_param("validation_size", 0.25)

print("参数总览:\n")
print(ae.param.overview())


参数总览:

  TabularX 参数总览

【Profile 阶段】
           text_unique_threshold =   200   #文本列判定：唯一值超过此数量 → 可能为文本列
               text_unique_ratio =   0.5   #文本列判定：唯一值占比（唯一值数/行数）超过此值 → 文本列
       int_categorical_threshold =    20   #整数列判定：唯一值数量 ≤ 此值 → 类别列
            scale_mean_threshold =   0.5   #Scale 判据：|mean| > 此值 → 需要标准化
             scale_std_threshold =   1.5   #Scale 判据：std > 此值 → 需要标准化
      high_cardinality_threshold =    50   #高基数判定：唯一值 > 此值 → high_cardinality
                  skew_threshold =   1.0   #偏斜判定：|skew| > 此值 → skewed
                 imbalance_ratio =   0.1   #类别不平衡：最小类别占比 < 此值 → imbalanced
          min_samples_regression =    10   #回归任务最少样本数（低于此值报错）
           min_samples_per_class =     2   #分类任务每类最少样本数（低于此值报错）

【Split 阶段】
                       test_size =   0.2   #测试集占比（占总样本的比例）
                 validation_size =  0.25   #验证集占比（相对于去掉测试集后剩余部分的比例）
                    random_state =    42   #随机种子（保证划分可复现）
               min_samples_split =     5   #训练/验证/测试集每集最少样本数

【P

## 一键运行

多分类任务会使用相同的流水线，SHAP 会为每个类别输出特征重要性。


In [4]:
ae.fit(X, y)

print("运行完成！")
import os
print(f"输出目录: {os.path.abspath(str(ae.output_dir))}")


[I 2026-09-12 22:41:44,397] A new study created in memory with name: random_forest_multiclass
[I 2026-09-12 22:41:44,399] A new study created in memory with name: catboost_multiclass
[I 2026-09-12 22:41:44,399] A new study created in memory with name: xgboost_multiclass
[I 2026-09-12 22:41:44,400] A new study created in memory with name: lightgbm_multiclass
[W 2026-09-12 22:41:44,734] Trial 0 failed with parameters: {'n_estimators': 200, 'max_depth': 15, 'learning_rate': 0.06504856968981275, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598} because of the following error: XGBoostError('[22:41:44] /__w/xgboost/xgboost/src/common/common.cu:16: /__w/xgboost/xgboost/src/common/cuda_dr_utils.cc: 28: cudaErrorInsufficientDriver: CUDA driver version is insufficient for CUDA runtime version\nStack trace:\n  [bt] (0) /home/frank/pyvenv/machine_learning_

运行完成！
输出目录: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro2-output


## 查看搜索结果


In [5]:
result = ae.result

print(f"任务类型: {result.searched.task_type}")
print(f"主指标: {result.searched.primary_metric}（{"越大越好" if result.searched.optimization_direction == "maximize" else "越小越好"}）\n")

print("=" * 70)
print(f"{"排名":>4} {"模型":>20} {"验证集":>10} {"测试集":>10}")
print("=" * 70)
for i, r in enumerate(result.searched.ranking, 1):
    valid = r.validation_metrics.get(result.searched.primary_metric, float("nan"))
    test = (r.test_metrics or {}).get(result.searched.primary_metric, float("nan"))
    print(f"{i:>4} {r.model_name:>20} {valid:>10.4f} {test:>10.4f}")

print(f"\n最佳模型: {result.searched.best_model_name}")


任务类型: multiclass
主指标: f1_macro（越大越好）

  排名                   模型        验证集        测试集
   1        random_forest     0.9333     0.9666
   2             catboost     0.9333     0.9333
   3             lightgbm     0.9666     0.8997

最佳模型: random_forest


## SHAP 可解释性

多分类时，SHAP 会展示每个类别的重要特征。Top-33 中每个类别都有对应的 SHAP 值。


In [6]:
shap = result.shap

if shap is not None:
    print(f"模型: {shap.model_name}")
    print(f"解释样本数: {shap.n_explain_samples}")
    print(f"Top-{len(shap.top_k_features)} 特征:\n{shap.top_k_features[:6]}")


模型: random_forest
解释样本数: 30
Top-4 特征:
['petal length (cm)', 'petal width (cm)', 'sepal length (cm)', 'sepal width (cm)']


## 总结

多分类流程与二分类完全一致，只需指定 `task="classification"`。TabularX 自动识别 Iris 为 multiclass，选择合适的主指标（f1_macro），并为每个类别生成 SHAP 解释。


In [7]:
from pathlib import Path

print("生成的文件:")
for name, p in result.report_paths.items():
    exists = "OK" if Path(p).exists() else "MISS"
    print(f"  [{exists}] {name}: {p}")

print(f"\n完整报告: {os.path.abspath(str(result.report_paths["full"]))}")


生成的文件:
  [OK] profile: intro2-output/reports/profile.html
  [OK] split: intro2-output/reports/split.html
  [OK] preprocess: intro2-output/reports/preprocess.html
  [OK] model_search: intro2-output/reports/model_search.html
  [OK] explanation: intro2-output/reports/explanation.html
  [OK] full: intro2-output/report.html

完整报告: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro2-output/report.html
